# 02 – Data Preprocessing & K-Fold Split

**Projekt:** Nike / Adidas / Converse cipőmárkák képosztályozása  
**Cél:** Az adatok tisztítása (SHA-256 duplikátum-szűrés), független Teszt készlet leválasztása, és **Stratified 5-Fold Keresztvalidációs felosztás** elkészítése a tanító adatsoron.

**Kimenet:** A képek elérési útját, címkéit és a `fold` beosztást tartalmazó CSV manifest (`../results/brand/dataset_with_folds.csv`).

## 1. Importok, Beállítások és Reprodukálhatóság

In [1]:
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import StratifiedKFold, train_test_split

# Paraméterek rögzítése
RANDOM_STATE = 42
K_FOLDS = 5
TEST_RATIO = 0.15

PROJECT_ROOT = Path("..").resolve()
DATASET_PATH = PROJECT_ROOT / "dataset" / "brand"
REPORT_PATH = PROJECT_ROOT / "results" / "brand"
REPORT_PATH.mkdir(parents=True, exist_ok=True)

CLASSES = ["nike", "adidas", "converse"]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print(f"Projekt gyökérkönyvtár: {PROJECT_ROOT}")
print(f"Bemeneti adatok: {DATASET_PATH}")

Projekt gyökérkönyvtár: C:\egyetem\5felev\Adatelemzés\hf\AMM_SZ_HF_cipofelismero
Bemeneti adatok: C:\egyetem\5felev\Adatelemzés\hf\AMM_SZ_HF_cipofelismero\dataset\brand


## 2. Képfájlok Összegyűjtése és Ellenőrzése

In [2]:
records = []
invalid_files = []

for label in CLASSES:
    class_dir = DATASET_PATH / label
    for path in sorted(class_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        try:
            with Image.open(path) as img:
                img.verify()
            with Image.open(path) as img:
                width, height = img.size
                image_format = img.format

            records.append(
                {
                    "path": str(path.resolve()),
                    "label": label,
                    "width": width,
                    "height": height,
                    "format": image_format,
                    "file_size_bytes": path.stat().st_size,
                }
            )
        except (UnidentifiedImageError, OSError, ValueError) as exc:
            invalid_files.append({"path": str(path.resolve()), "label": label, "error": str(exc)})

data = pd.DataFrame(records)
print(f"Érvényes képek száma: {len(data)}")
display(data.groupby("label").size().rename("valid_images").to_frame())

Érvényes képek száma: 600


,valid_images
label,
adidas,210
converse,195
nike,195


## 3. SHA-256 Duplikátumok Kiszűrése

In [3]:
def sha256_file(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(file_path, "rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

data["sha256"] = data["path"].apply(sha256_file)

# Címkézési konfliktusok szűrése
hash_label_counts = data.groupby("sha256")["label"].nunique()
conflict_hashes = set(hash_label_counts[hash_label_counts > 1].index)
without_conflicts = data[~data["sha256"].isin(conflict_hashes)].copy()

# Azonos tartalmú duplikátumok eltávolítása
deduplicated = without_conflicts.drop_duplicates(subset="sha256", keep="first").copy()

data = deduplicated.reset_index(drop=True)
print(f"Tisztított adathalmaz mérete: {len(data)} kép")

Tisztított adathalmaz mérete: 600 kép


## 4. Független Teszt Készlet Leválasztása és Stratified K-Fold Split

In [5]:
# 1. Lépés: Független Holdout Teszt készlet leválasztása (15%)
train_val_df, test_df = train_test_split(
    data,
    test_size=TEST_RATIO,
    random_state=RANDOM_STATE,
    stratify=data["label"]
)

# 2. Lépés: Stratified K-Fold felosztás a tanító-validációs adatsoron
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)

train_val_df = train_val_df.copy()
train_val_df["fold"] = -1

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_val_df, train_val_df["label"])):
    train_val_df.iloc[val_idx, train_val_df.columns.get_loc("fold")] = fold_idx

# Teszt készlet megjelölése (fold = -1)
test_df = test_df.copy()
test_df["fold"] = -1
test_df["is_test"] = True
train_val_df["is_test"] = False

# Teljes manifest összefűzése
full_manifest = pd.concat([train_val_df, test_df], ignore_index=True)

## 5. Ellenőrzés és Manifest Mentése

In [6]:
print("\n--- Felosztás összegzése ---")
print(f"Teljes tisztított adathalmaz: {len(full_manifest)} kép")
print(f"Holdout Teszt halmaz (fold = -1): {len(test_df)} kép")
print(f"Keresztvalidációs adatsor (5 fold-ra osztva): {len(train_val_df)} kép")

display_df = pd.crosstab(full_manifest["fold"], full_manifest["label"])
display_df.index = [f"Fold {i}" if i >= 0 else "Holdout Test" for i in display_df.index]
display(display_df)

# Manifest és osztálynevek elmentése
manifest_path = REPORT_PATH / "dataset_with_folds.csv"
full_manifest.to_csv(manifest_path, index=False)
print(f"\nSikeresen elmentve: {manifest_path}")

with open(REPORT_PATH / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, ensure_ascii=False, indent=4)


--- Felosztás összegzése ---
Teljes tisztított adathalmaz: 600 kép
Holdout Teszt halmaz (fold = -1): 90 kép
Keresztvalidációs adatsor (5 fold-ra osztva): 510 kép


label,adidas,converse,nike
Holdout Test,32,29,29
Fold 0,35,34,33
Fold 1,36,33,33
Fold 2,36,33,33
Fold 3,36,33,33
Fold 4,35,33,34



Sikeresen elmentve: C:\egyetem\5felev\Adatelemzés\hf\AMM_SZ_HF_cipofelismero\results\brand\dataset_with_folds.csv
